# Comparación de Estrategias Federadas vs Proactive Forest Base

## Objetivo
Comparar las métricas (Accuracy y Macro-F1) de cada estrategia federada (S1-S7 + PW) con los resultados base de Proactive Forest reportados por Nayma, usando **3 clientes** y promediando las métricas de los 3 clientes.

## Datasets
Car, Iris, Letter, Nursery, Optdigits, Sonar, Spambase, Vowel

## Configuración fija
- **n_clients = 3**
- **seed = 42**
- **n_estimators = 100**
- **distribution = iid**
- **alpha_pf = 0.45**
- **t_max = 100** (S2-S7)
- **local_weight = 0.5**
- **f1_weight = 0.6** (S4, S7, PW)
- **window_size = 7** (PW)
- **max_rounds = 15** (PW)
- **convergence_threshold = 0.002** (PW)

---
## 0. Imports y configuración común

In [1]:
import sys
from pathlib import Path

# Notebook está en src/interfaces/notebooks/ → buscar project root
ROOT = Path.cwd().resolve()
while not (ROOT / 'src').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

print(f'Project root: {ROOT}')
assert (ROOT / 'src').exists(), f'No se encontró src/ en {ROOT}'

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from src.application.orchestrators import FLEXOrchestrator
from src.domain.dataset.base_adapter import DatasetSplit

SEED = 42
N_CLIENTS = 3
N_ESTIMATORS = 100
ALPHA_PF = 0.45
T_MAX = 100
LOCAL_WEIGHT = 0.5
F1_WEIGHT = 0.6

np.random.seed(SEED)

Project root: C:\Users\Adrián Rodríguez\Documents\! Study 📝\📊 KDD 🤖🧠\! Federated learning\federated_proactive_forest


---
## 1. Dataset Loaders

In [2]:
def load_car():
    df = pd.read_csv(ROOT / 'data' / 'car.csv')
    fcols = [c for c in df.columns if c != 'class']
    enc = OrdinalEncoder()
    X = enc.fit_transform(df[fcols]).astype(float)
    le = LabelEncoder(); y = le.fit_transform(df['class'])
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=.2, random_state=SEED, stratify=y)
    return DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,
                        feature_names=fcols, class_names=[str(c) for c in le.classes_], dataset_name='car')

def load_iris():
    from sklearn.datasets import load_iris
    data = load_iris()
    X_tr, X_te, y_tr, y_te = train_test_split(data.data, data.target, test_size=.2, random_state=SEED, stratify=data.target)
    sc = StandardScaler(); X_tr = sc.fit_transform(X_tr); X_te = sc.transform(X_te)
    return DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,
                        feature_names=list(data.feature_names), class_names=[str(c) for c in data.target_names], dataset_name='iris')

def load_letter():
    df = pd.read_csv(ROOT / 'data' / 'letter.csv')
    fcols = [c for c in df.columns if c != 'class']
    X = df[fcols].values.astype(float)
    le = LabelEncoder(); y = le.fit_transform(df['class'])
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=.2, random_state=SEED, stratify=y)
    sc = StandardScaler(); X_tr = sc.fit_transform(X_tr); X_te = sc.transform(X_te)
    return DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,
                        feature_names=fcols, class_names=[str(c) for c in le.classes_], dataset_name='letter')

def load_nursery():
    df = pd.read_csv(ROOT / 'data' / 'nursery.csv')
    fcols = [c for c in df.columns if c != 'class']
    enc = OrdinalEncoder()
    X = enc.fit_transform(df[fcols]).astype(float)
    le = LabelEncoder(); y = le.fit_transform(df['class'])
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=.2, random_state=SEED, stratify=y)
    return DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,
                        feature_names=fcols, class_names=[str(c) for c in le.classes_], dataset_name='nursery')

def load_optdigits():
    df = pd.read_csv(ROOT / 'data' / 'optdigits.csv')
    fcols = [c for c in df.columns if c != 'class']
    X = df[fcols].values.astype(float)
    le = LabelEncoder(); y = le.fit_transform(df['class'])
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=.2, random_state=SEED, stratify=y)
    sc = StandardScaler(); X_tr = sc.fit_transform(X_tr); X_te = sc.transform(X_te)
    return DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,
                        feature_names=fcols, class_names=[str(c) for c in le.classes_], dataset_name='optdigits')

def load_sonar():
    df = pd.read_csv(ROOT / 'data' / 'sonar.csv')
    fcols = [c for c in df.columns if c != 'Class']
    X = df[fcols].values.astype(float)
    le = LabelEncoder(); y = le.fit_transform(df['Class'])
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=.2, random_state=SEED, stratify=y)
    sc = StandardScaler(); X_tr = sc.fit_transform(X_tr); X_te = sc.transform(X_te)
    return DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,
                        feature_names=fcols, class_names=[str(c) for c in le.classes_], dataset_name='sonar')

def load_spambase():
    df = pd.read_csv(ROOT / 'data' / 'spambase.csv')
    fcols = [c for c in df.columns if c != 'class']
    X = df[fcols].values.astype(float)
    le = LabelEncoder(); y = le.fit_transform(df['class'])
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=.2, random_state=SEED, stratify=y)
    sc = StandardScaler(); X_tr = sc.fit_transform(X_tr); X_te = sc.transform(X_te)
    return DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,
                        feature_names=fcols, class_names=[str(c) for c in le.classes_], dataset_name='spambase')

def load_vowel():
    df = pd.read_csv(ROOT / 'data' / 'vowel.csv')
    drop_cols = ['Train or Test', 'Speaker Number', 'Sex']
    use_cols = [c for c in df.columns if c not in drop_cols and c != 'Class']
    X = df[use_cols].values.astype(float)
    le = LabelEncoder(); y = le.fit_transform(df['Class'])
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=.2, random_state=SEED, stratify=y)
    sc = StandardScaler(); X_tr = sc.fit_transform(X_tr); X_te = sc.transform(X_te)
    return DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,
                        feature_names=use_cols, class_names=[str(c) for c in le.classes_], dataset_name='vowel')

DATASETS = {
    'car': load_car, 'iris': load_iris, 'letter': load_letter,
    'nursery': load_nursery, 'optdigits': load_optdigits,
    'sonar': load_sonar, 'spambase': load_spambase, 'vowel': load_vowel,
}

---
## 2. Función de experimento genérica

In [3]:
def build_config(strategy, n_clients=N_CLIENTS, n_estimators=N_ESTIMATORS, seed=SEED):
    """Construye config con los parámetros unificados para cualquier estrategia."""
    cfg = {
        'federation': {'n_clients': n_clients, 'distribution': 'iid', 'seed': seed},
        'model': {
            'n_estimators': n_estimators, 'alpha': ALPHA_PF,
            'split_criterion': 'entropy', 'use_progressive_stopping': True,
            'convergence': 0.002, 'episode_size': 5, 'verbose': False,
        },
        'aggregation': {'strategy': strategy, 't_max': T_MAX},
        'prediction': {'local_weight': LOCAL_WEIGHT, 'global_weight': 1.0 - LOCAL_WEIGHT},
        'verbose': False, 'seed': seed,
    }

    # S4, S7, PW → f1_weight
    if strategy in ('S4', 'S7', 'PW'):
        cfg['aggregation']['f1_weight'] = F1_WEIGHT
        cfg['aggregation']['pcd_weight'] = 1.0 - F1_WEIGHT

    # PW → parámetros exclusivos
    if strategy == 'PW':
        cfg['aggregation']['window_size'] = 7
        cfg['aggregation']['max_rounds'] = 15
        cfg['aggregation']['convergence_threshold'] = 0.002

    return cfg


def run_experiment(dataset_name, strategy):
    """Ejecuta una combinación dataset × estrategia y devuelve dict con métricas por cliente."""
    ds = DATASETS[dataset_name]()
    cfg = build_config(strategy)
    np.random.seed(SEED)

    orch = FLEXOrchestrator.from_config(cfg)
    orch.setup_federation(ds, seed=SEED)
    results = orch.run_federated_round()

    # Métricas por cliente (híbridas)
    client_metrics = {}
    for cid, preds in results.client_hybrid_predictions.items():
        acc = accuracy_score(results.y_test, preds)
        f1 = f1_score(results.y_test, preds, average='macro', zero_division=0)
        client_metrics[cid] = {'accuracy': acc, 'macro_f1': f1}

    return {
        'dataset': dataset_name,
        'strategy': strategy,
        'global_accuracy': results.global_accuracy,
        'global_macro_f1': results.global_macro_f1,
        'client_metrics': client_metrics,
        'n_trees_global': results.n_trees_global,
    }

---
## 3. Ejecutar cada estrategia (celda independiente por si acaso)

> **Nota:** Cada celda ejecuta la estrategia para **todos los datasets**. Los resultados se acumulan en `all_results`.

In [4]:
# Celda compartida para almacenar resultados
all_results = {}  # {(dataset, strategy): result_dict}

In [5]:
# ── S1: Simple Pool ──────────────────────────────────────────────────────
STRATEGY = 'S1'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    print(f"F1={res['global_macro_f1']:.4f}")
print(f"✅ {STRATEGY} completada.")


🚀 Ejecutando S1
  ▶ car... 
🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 54 árboles entrenados
  client_1: 76 árboles entrenados
  client_2: 37 árboles entrenados
  TOTAL: 167 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S1)
  client_0: 54/54 árboles seleccionados
  client_1: 76/76 árboles seleccionados
  client_2: 37/37 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 167 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 54 locales + 113 globales externos (54 propios excluidos) = 167 árboles
  client_1: 76 locales + 91 globales externos (76 propios excluidos) = 167 árboles
  client_2: 37 locales + 130 globales externos (37 propios excluidos) = 167 árboles

F1=0.9025
  ▶ iris... 
🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 16 árboles entrenados
  client_1: 38 árboles entrenados
  client_2: 38 árboles entrenados
  TOTAL: 92 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S1)
  client_0: 16/16 árboles seleccionados
  client_1: 38/38 árboles seleccionados
  client_2:

In [6]:
# ── S2: Global Accuracy + PF ─────────────────────────────────────────────
STRATEGY = 'S2'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    print(f"F1={res['global_macro_f1']:.4f}")
print(f"✅ {STRATEGY} completada.")


🚀 Ejecutando S2
  ▶ car... 
🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 48 árboles entrenados
  client_1: 43 árboles entrenados
  client_2: 24 árboles entrenados
  TOTAL: 115 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S2)
  client_0: 48/48 árboles seleccionados
  client_1: 43/43 árboles seleccionados
  client_2: 24/24 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 115 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 48 locales + 67 globales externos (48 propios excluidos) = 115 árboles
  client_1: 43 locales + 72 globales externos (43 propios excluidos) = 115 árboles
  client_2: 24 locales + 91 globales externos (24 propios excluidos) = 115 árboles

F1=0.9302
  ▶ iris... 
🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 16 árboles entrenados
  client_1: 16 árboles entrenados
  client_2: 37 árboles entrenados
  TOTAL: 69 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S2)
  client_0: 16/16 árboles seleccionados
  client_1: 16/16 árboles seleccionados
  client_2: 3

In [7]:
# ── S3: Global Macro-F1 + PF ─────────────────────────────────────────────
STRATEGY = 'S3'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    print(f"F1={res['global_macro_f1']:.4f}")
print(f"✅ {STRATEGY} completada.")


🚀 Ejecutando S3
  ▶ car... 
🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 16 árboles entrenados
  client_1: 33 árboles entrenados
  client_2: 16 árboles entrenados
  TOTAL: 65 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S3)
  client_0: 16/16 árboles seleccionados
  client_1: 33/33 árboles seleccionados
  client_2: 16/16 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 65 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 16 locales + 49 globales externos (16 propios excluidos) = 65 árboles
  client_1: 33 locales + 32 globales externos (33 propios excluidos) = 65 árboles
  client_2: 16 locales + 49 globales externos (16 propios excluidos) = 65 árboles

F1=0.9220
  ▶ iris... 
🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 9 árboles entrenados
  client_1: 18 árboles entrenados
  client_2: 46 árboles entrenados
  TOTAL: 73 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S3)
  client_0: 9/9 árboles seleccionados
  client_1: 18/18 árboles seleccionados
  client_2: 46/46 árb

KeyboardInterrupt: 

In [ ]:
# ── S4: Global F1 + PCD + PF ─────────────────────────────────────────────
STRATEGY = 'S4'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    print(f"F1={res['global_macro_f1']:.4f}")
print(f"✅ {STRATEGY} completada.")

In [ ]:
# ── S5: Per-Client Accuracy + PF ─────────────────────────────────────────
STRATEGY = 'S5'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    print(f"F1={res['global_macro_f1']:.4f}")
print(f"✅ {STRATEGY} completada.")

In [ ]:
# ── S6: Per-Client Macro-F1 + PF ─────────────────────────────────────────
STRATEGY = 'S6'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    print(f"F1={res['global_macro_f1']:.4f}")
print(f"✅ {STRATEGY} completada.")

In [ ]:
# ── S7: Per-Client F1 + PCD + PF ─────────────────────────────────────────
STRATEGY = 'S7'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    print(f"F1={res['global_macro_f1']:.4f}")
print(f"✅ {STRATEGY} completada.")

In [ ]:
# ── PW: Progressive Windows ──────────────────────────────────────────────
STRATEGY = 'PW'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    print(f"F1={res['global_macro_f1']:.4f}")
print(f"✅ {STRATEGY} completada.")

---
## 4. Tabla comparativa final

Se calcula el **promedio de Accuracy y F1 de los 3 clientes** para cada combinación dataset × estrategia.

In [8]:
# ── Resultados base de Nayma (Proactive Forest) ──────────────────────────
NAYMA_RESULTS = {
    'car':       {'accuracy': 0.976625780, 'f1': 0.945782},
    'iris':      {'accuracy': 0.956000000, 'f1': 0.954981},
    'letter':    {'accuracy': 0.965045493, 'f1': 0.965253},
    'nursery':   {'accuracy': 0.995910870, 'f1': 0.954848},
    'optdigits': {'accuracy': 0.983235639, 'f1': 0.982218},
    'sonar':     {'accuracy': 0.848298701, 'f1': 0.823483},
    'spambase':  {'accuracy': 0.953879759, 'f1': 0.952755},
    'vowel':     {'accuracy': 0.971919192, 'f1': 0.968468},
}

STRATEGIES = ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'PW']
DATASET_ORDER = ['car', 'iris', 'letter', 'nursery', 'optdigits', 'sonar', 'spambase', 'vowel']

In [9]:
# ── Construir tabla comparativa ──────────────────────────────────────────
rows = []

for ds in DATASET_ORDER:
    row = {'BD': ds.capitalize()}

    # Columna PF (Nayma)
    row['Accuracy_PF'] = round(NAYMA_RESULTS[ds]['accuracy'], 6)
    row['F1_PF'] = round(NAYMA_RESULTS[ds]['f1'], 6)

    # Columnas por estrategia
    for strat in STRATEGIES:
        key = (ds, strat)
        if key in all_results:
            res = all_results[key]
            # Promedio de los 3 clientes
            client_accs = [m['accuracy'] for m in res['client_metrics'].values()]
            client_f1s = [m['macro_f1'] for m in res['client_metrics'].values()]
            row[f'Accuracy_{strat}'] = round(np.mean(client_accs), 6)
            row[f'F1_{strat}'] = round(np.mean(client_f1s), 6)
        else:
            row[f'Accuracy_{strat}'] = None
            row[f'F1_{strat}'] = None

    rows.append(row)

df_comparison = pd.DataFrame(rows)

# Reordenar columnas: BD, PF, luego cada estrategia
col_order = ['BD', 'Accuracy_PF', 'F1_PF']
for strat in STRATEGIES:
    col_order += [f'Accuracy_{strat}', f'F1_{strat}']
df_comparison = df_comparison[col_order]

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', lambda x: f'{x:.6f}' if pd.notna(x) else '')

print(df_comparison.to_string(index=False))

       BD  Accuracy_PF    F1_PF  Accuracy_S1    F1_S1  Accuracy_S2    F1_S2  Accuracy_S3    F1_S3 Accuracy_S4 F1_S4 Accuracy_S5 F1_S5 Accuracy_S6 F1_S6 Accuracy_S7 F1_S7 Accuracy_PW F1_PW
      Car     0.976626 0.945782     0.222543 0.091017     0.222543 0.091017     0.222543 0.091017        None  None        None  None        None  None        None  None        None  None
     Iris     0.956000 0.954981     0.333333 0.166667     0.333333 0.166667     0.333333 0.166667        None  None        None  None        None  None        None  None        None  None
   Letter     0.965045 0.965253     0.039500 0.002923     0.039500 0.002923          NaN      NaN        None  None        None  None        None  None        None  None        None  None
  Nursery     0.995911 0.954848     0.333333 0.125000     0.333333 0.125000          NaN      NaN        None  None        None  None        None  None        None  None        None  None
Optdigits     0.983236 0.982218     0.098754 0.017976     0.

In [10]:
# ── Guardar tabla como CSV y Excel ───────────────────────────────────────
out_dir = ROOT / 'results' / 'comparison_vs_nayma'
out_dir.mkdir(parents=True, exist_ok=True)

df_comparison.to_csv(out_dir / 'comparison_table.csv', index=False)
df_comparison.to_excel(out_dir / 'comparison_table.xlsx', index=False)
print(f"\n✅ Tabla guardada en: {out_dir}")


✅ Tabla guardada en: C:\Users\Adrián Rodríguez\Documents\! Study 📝\📊 KDD 🤖🧠\! Federated learning\federated_proactive_forest\results\comparison_vs_nayma


In [11]:
# ── Tabla de diferencias vs Nayma (estrategia - PF) ─────────────────────
diff_rows = []

for ds in DATASET_ORDER:
    row = {'BD': ds.capitalize()}
    for strat in STRATEGIES:
        key = (ds, strat)
        if key in all_results:
            res = all_results[key]
            client_accs = [m['accuracy'] for m in res['client_metrics'].values()]
            client_f1s = [m['macro_f1'] for m in res['client_metrics'].values()]
            avg_acc = np.mean(client_accs)
            avg_f1 = np.mean(client_f1s)
            row[f'ΔAcc_{strat}'] = round(avg_acc - NAYMA_RESULTS[ds]['accuracy'], 6)
            row[f'ΔF1_{strat}'] = round(avg_f1 - NAYMA_RESULTS[ds]['f1'], 6)
        else:
            row[f'ΔAcc_{strat}'] = None
            row[f'ΔF1_{strat}'] = None
    diff_rows.append(row)

diff_cols = ['BD']
for strat in STRATEGIES:
    diff_cols += [f'ΔAcc_{strat}', f'ΔF1_{strat}']

df_diff = pd.DataFrame(diff_rows)[diff_cols]
print(df_diff.to_string(index=False))

df_diff.to_csv(out_dir / 'comparison_diff_vs_PF.csv', index=False)
print(f"\n✅ Diferencias guardadas en: {out_dir / 'comparison_diff_vs_PF.csv'}")

       BD   ΔAcc_S1    ΔF1_S1   ΔAcc_S2    ΔF1_S2   ΔAcc_S3    ΔF1_S3 ΔAcc_S4 ΔF1_S4 ΔAcc_S5 ΔF1_S5 ΔAcc_S6 ΔF1_S6 ΔAcc_S7 ΔF1_S7 ΔAcc_PW ΔF1_PW
      Car -0.754082 -0.854765 -0.754082 -0.854765 -0.754082 -0.854765    None   None    None   None    None   None    None   None    None   None
     Iris -0.622667 -0.788314 -0.622667 -0.788314 -0.622667 -0.788314    None   None    None   None    None   None    None   None    None   None
   Letter -0.925545 -0.962330 -0.925545 -0.962330       NaN       NaN    None   None    None   None    None   None    None   None    None   None
  Nursery -0.662578 -0.829848 -0.662578 -0.829848       NaN       NaN    None   None    None   None    None   None    None   None    None   None
Optdigits -0.884481 -0.964242 -0.884481 -0.964242       NaN       NaN    None   None    None   None    None   None    None   None    None   None
    Sonar -0.324489 -0.479733 -0.324489 -0.479733       NaN       NaN    None   None    None   None    None   None    None   None 